In [5]:
import numpy as np
import CoRT_builder
import utils
from sklearn.linear_model import Lasso
from scipy.stats import norm

In [6]:
n_target = 30
n_source = 40
p = 100
K = 7
Ka = 2
h = 30
alpha = 0.05
T = 5
s_len = 30
s_vector = [0.5] * s_len

CoRT_model = CoRT_builder.CoRT(0)
para_results_storage = []
iterations = 1000

for i in range(iterations):
    target_data, source_data = CoRT_model.gen_data(n_target, n_source, p, K, Ka, h, s_vector, s_len, "AR")
    X_target = target_data["X"]
    y_target = target_data["y"]
    X_train_list = [X_target]
    y_train_list = [y_target]
    for k in range(K):
        X_train_list.append(source_data[k]["X"])
        y_train_list.append(source_data[k]["y"])
    X_combined = np.vstack(X_train_list)
    y_combined = np.vstack(y_train_list)
    n = X_combined.shape[0]
    lamda = np.sqrt(np.log(p) / n)
    model = Lasso(alpha=lamda, fit_intercept=False, tol=1e-12, max_iter=1000000)
    model.fit(X_combined, y_combined.ravel())

    beta_hat_target = model.coef_[-p:]
    M_obs = np.array([i for i, b in enumerate(beta_hat_target) if b != 0])
    if len(M_obs) == 0:
        print(f"Iteration {iter}: Lasso selected no features. Skipping.")
        continue

    j = np.random.choice(len(M_obs))
    selected_feature_index = M_obs[j]

    X_active, X_inactive = utils.get_active_X(beta_hat_target, X_combined)
    etaj, etajTy = utils.construct_test_statistic(y_combined, j, X_active)
    Sigma = np.eye(n)
    tn_sigma = (np.sqrt(etaj.T @ Sigma @ etaj)).item()

    p_value = 2 * (1 - norm.cdf(abs(etajTy), loc = 0, scale = tn_sigma))

    is_signal = (selected_feature_index < s_len) 
    result_dict = {
        "p_value": p_value,
        "is_signal": is_signal,
        "feature_idx": selected_feature_index
    }
    # print(f"is_signal : {result_dict['is_signal']}, p_values[{i}]: {result_dict['p_value']}")
    para_results_storage.append(result_dict)
    if (i % 50 == 0):
        print(f"Iteration {i}")

is_signal_cases = [r for r in para_results_storage if r['is_signal']]
not_signal_cases = [r for r in para_results_storage if not r['is_signal']]

false_positives = sum(1 for c in not_signal_cases if c['p_value'] <= alpha)
print(f"len not_signal_cases : {len(not_signal_cases)}")
print(f"false_positives: {false_positives}")
fpr = false_positives / len(not_signal_cases)
print(f"FPR: {fpr:.4f} (Target: {alpha})")
print("\n")
true_positives = sum(1 for r in is_signal_cases if r['p_value'] <= alpha)
print(f"len signal_cases : {len(is_signal_cases)}")
print(f"true_positives: {true_positives}")
tpr = true_positives / len(is_signal_cases)
print(f"TPR: {tpr:.4f}")


Iteration 0
Iteration 50
Iteration 100
Iteration 150
Iteration 200
Iteration 250
Iteration 300
Iteration 350
Iteration 400
Iteration 450
Iteration 500
Iteration 550
Iteration 600
Iteration 650
Iteration 700
Iteration 750
Iteration 800
Iteration 850
Iteration 900
Iteration 950
len not_signal_cases : 728
false_positives: 655
FPR: 0.8997 (Target: 0.05)


len signal_cases : 272
true_positives: 255
TPR: 0.9375
